# Ensemble Learning with Scikit-Learn, CatBoost, and XGBoost


1. **Bagging (Bootstrap Aggregating)** — *Variance reduction via bootstrap sampling with replacement and Out-of-Bag (OOB) scoring*
2. **Boosting (AdaBoost)** — *Sequential bias reduction via adaptive sample re-weighting and decision stumps*
3. **Stacking (Stacked Generalization)** — *Heterogeneous multi-model blending with out-of-fold meta-learning*
4. **Random Forest** — *De-correlated decision forest via the Random Subspace method ($m = \sqrt{p}$)*
5. **Gradient Boosting Decision Trees (GBDT)** — *Sequential gradient descent in function space fitting pseudo-residuals*
6. **CatBoost (Categorical Boosting)** — *Symmetric oblivious trees, ordered target encoding, and fast CPU/GPU inference*
7. **XGBoost (Extreme Gradient Boosting)** — *2nd-order Taylor loss approximation, exact greedy split finding, and L1/L2 regularization*
8. **Comprehensive Ensemble Benchmark & Comparison** — *Synthesizing all 7 paradigms on an identical test split*
9. **Comparative Architecture Summary & Practical Strategy Guide** — *Selection Matrix, Workflow Hierarchy & Pitfalls*

---

### 🔄 Standard 8-Step Pedagogical Workflow:
To maximize comparative clarity, every ensemble paradigm is implemented using the standardized 8-step recipe:
1. **Importing** — Specific ensemble estimators, base learners, and metric suites.
2. **Loading Dataset** — Ingesting the Wisconsin Breast Cancer benchmark.
3. **Split** — Stratified train/test partitioning to guarantee identical class distributions.
4. **Train (Model Setup)** — Configuring hyperparameters (`n_estimators`, `learning_rate`, `subsample`, tree depth).
5. **Fit** — Executing parallel bagging or sequential boosting training.
6. **Predict** — Voting or weighted score aggregation across all base estimators.
7. **Evaluation** — Calculating test accuracy, confusion matrix, OOB scores, and deviance.
8. **Visualization** — Multi-panel publication-grade diagnostic plots (learning curves, feature importances, staged accuracy).


---
# 1. Bagging (Bootstrap Aggregating)

Bagging is a parallel ensemble paradigm designed to **drastically reduce variance** without increasing bias by training multiple unconstrained, high-variance base estimators (e.g., fully grown decision trees) on independent bootstrap resamples of the training data.

### 📐 The Mathematics of Variance Reduction:
Given $M$ independent identically distributed estimators each with variance $\sigma^2$ and pairwise correlation $\rho$, the variance of their ensemble average $\bar{X} = \frac{1}{M} \sum_{m=1}^M X_m$ is:
$$\text{Var}(\bar{X}) = \rho \sigma^2 + \frac{1 - \rho}{M} \sigma^2$$
- If base models are completely independent ($\rho = 0$), the ensemble variance drops linearly with $M$: $\text{Var}(\bar{X}) = \frac{\sigma^2}{M}$.
- In practice, since all base models draw from the same parent dataset, $\rho > 0$. Bagging drives the second term $\frac{1-\rho}{M} \sigma^2 \to 0$, stabilizing erratic individual predictions.

> 🍋 **Layman's Analogy:**  
> Imagine asking a single doctor for a rare medical diagnosis. If that doctor is having a bad day or made a single mistake in medical school, their diagnosis might be completely wrong (high variance).  
> Instead, you consult **100 independent doctors** who each review different clinical notes, and you take a majority vote! Even if 20 doctors make individual mistakes, the collective majority vote is extraordinarily reliable!

### 🔬 Mathematical Note: The Out-of-Bag (OOB) Phenomenon
When drawing $N$ samples with replacement from a dataset of size $N$, the probability that a specific sample is NOT chosen in a single draw is $1 - \frac{1}{N}$. Across $N$ draws:
$$\lim_{N \to \infty} \left(1 - \frac{1}{N}\right)^N = \frac{1}{e} \approx 0.368$$
This means $\approx 36.8\%$ of samples are omitted from each tree's training set! These omitted samples form an **Out-of-Bag (OOB)** validation set, giving an unbiased estimate of generalization error **for free** without cross-validation.

### 🌍 Real-World Use Cases:
- **Stabilizing High-Variance Predictors:** Smoothing noisy, unpruned decision trees in clinical risk scoring, manufacturing defect triage, and remote-sensing land classification.
- **Data-Scarce Validation:** Estimating true generalization accuracy via `oob_score=True` when datasets are small (`569` biopsies in Breast Cancer) and withholding an extra validation split is expensive.

### ⚠️ Pitfalls of Bagging:
- **Does Not Reduce Bias:** If your base learner is too rigid (e.g., Linear Regression or a 1-split decision stump), averaging 100 underfitting models yields an equally underfitting ensemble.
- **Correlated Trees When One Feature Dominates:** Because every tree still evaluates all 30 Breast Cancer features at every split, dominant predictors like `worst perimeter` or `worst concave points` get chosen near the root of almost every tree ($\rho$ remains high—which Random Forest fixes!).
- **Loss of Single-Tree Interpretability:** You can print and read 1 decision tree; you cannot visually trace 100 overlapping bootstrap trees.

### Step 1: Importing (Bagging)

> 💡 *Beginner Tip:* We import `BaggingClassifier` and `DecisionTreeClassifier` from Scikit-Learn along with classification assessment metrics.

In [1]:
# 1. Importing libraries for Bagging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11



### Step 2: Loading Dataset

> 💡 *Real-World Context:* We load the Wisconsin Breast Cancer benchmark (569 biopsies, 30 features) to compare variance reduction across unstable decision trees.

In [2]:
# 2. Loading Dataset
cancer_bag = load_breast_cancer(as_frame=True)
df_cancer_bag = cancer_bag.frame

X_bag = df_cancer_bag[cancer_bag.feature_names]
y_bag = df_cancer_bag['target']

print(f"Dataset Shape: {X_bag.shape} (569 samples, 30 features)")
print("Classes:", cancer_bag.target_names)
display(df_cancer_bag.head(3))


Dataset Shape: (569, 30) (569 samples, 30 features)
Classes: ['malignant' 'benign']


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0


### Step 3: Split (Stratified Partitioning — Why No Scaling Is Needed)

> ⚠️ *Critical Engineering Concept — `stratify=y_bag` & Tree Scale Invariance on Breast Cancer:*  
> - **`stratify=y_bag`:** Preserves the exact `37.3%` malignant / `62.7%` benign ratio across `426` training patients and `143` test patients (`test_size=0.25`).
> - **Why is `StandardScaler` omitted here?** Even though `worst area` spans `185.2–4254.0` while `smoothness error` spans `0.0017–0.0311`, `DecisionTreeClassifier` only checks threshold orderings (`worst area <= 880.8`). Tree splits are completely invariant to monotonic rescaling!

In [3]:
# 3. Train/Test Split
X_train_bag, X_test_bag, y_train_bag, y_test_bag = train_test_split(
    X_bag, y_bag, test_size=0.25, random_state=42, stratify=y_bag
)

print(f"Training samples: {len(X_train_bag)} | Testing samples: {len(X_test_bag)}")


Training samples: 426 | Testing samples: 143


### Step 4: Train (Model Setup & Hyperparameters)

> 💡 *Hyperparameter Intuition — Why `n_estimators=100`, `oob_score=True`, & `n_jobs=-1` Are Used:*  
> - **`estimator=DecisionTreeClassifier()`:** Uses fully grown, unpruned trees (`max_depth=None`) deliberately—high variance, low bias—so Bagging has variance to crush.
> - **`n_estimators=100`:** Trains 100 distinct bootstrap trees to stabilize the majority vote.
> - **`oob_score=True`:** Evaluates each of the 426 training patients exclusively on the ~37 trees that *did not* draw that patient during bootstrap sampling.
> - **`n_jobs=-1`:** Parallelizes tree fitting across all available CPU cores since every bootstrap tree trains independently.

In [6]:
# 4. Train: Model Initialization
single_tree = DecisionTreeClassifier(random_state=42)

bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

print("Baseline Single Tree and Bagging Ensemble initialized.")


Baseline Single Tree and Bagging Ensemble initialized.


### Step 5: Fit (Parallel Bootstrap Training)

> 💡 *Mathematical Insight:*  
> During `.fit()`, `single_tree` sees the exact `426` training patients once, while `bag_clf` draws `100` independent bootstrap replicas of size `426` with replacement and fits 100 independent unpruned trees in parallel.

In [7]:
# 5. Fit: Model Training
single_tree.fit(X_train_bag, y_train_bag)
bag_clf.fit(X_train_bag, y_train_bag)

print(f"Bagging ensemble fitted! OOB Score: {bag_clf.oob_score_:.4f}")


Bagging ensemble fitted! OOB Score: 0.9554


### Step 6: Predict (Majority Voting Inference)

> 💡 *Beginner Tip:*  
> Test predictions for each of the `143` holdout biopsies are generated by averaging class probability estimates across all `100` constituent trees into a stabilized final diagnosis.

In [8]:
# 6. Predict: Model Inference
y_pred_single = single_tree.predict(X_test_bag)
y_pred_bag = bag_clf.predict(X_test_bag)

print("First 10 Bagging Predictions:", y_pred_bag[:10])


First 10 Bagging Predictions: [1 0 0 1 1 0 0 0 0 1]


### Step 7: Quantitative Evaluation (Bagging vs. Single Tree)

> 💡 *How to Interpret Ensemble Metrics:*  
> Compare `acc_single` (1 unpruned tree) directly against `acc_bag` (100 bagged trees) and `bag_clf.oob_score_`: notice the **Variance Reduction Gain** without changing a single hyperparameter of the underlying tree!

In [9]:
# 7. Evaluation
acc_single = accuracy_score(y_test_bag, y_pred_single)
acc_bag = accuracy_score(y_test_bag, y_pred_bag)
cm_bag = confusion_matrix(y_test_bag, y_pred_bag)

print("=" * 60)
print("BAGGING ENSEMBLE EVALUATION REPORT")
print("=" * 60)
print(f"Single Decision Tree Accuracy:   {acc_single:.4f} ({acc_single*100:.2f}%)")
print(f"Bagging Ensemble (100 Trees):    {acc_bag:.4f} ({acc_bag*100:.2f}%)")
print(f"Out-of-Bag (OOB) Accuracy Score: {bag_clf.oob_score_:.4f} ({bag_clf.oob_score_*100:.2f}%)")
print(f"Variance Reduction Gain:         +{(acc_bag - acc_single)*100:.2f}%")
print("=" * 60)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test_bag, y_pred_bag, target_names=cancer_bag.target_names))


BAGGING ENSEMBLE EVALUATION REPORT
Single Decision Tree Accuracy:   0.9231 (92.31%)
Bagging Ensemble (100 Trees):    0.9510 (95.10%)
Out-of-Bag (OOB) Accuracy Score: 0.9554 (95.54%)
Variance Reduction Gain:         +2.80%
CLASSIFICATION REPORT:
              precision    recall  f1-score   support

   malignant       0.94      0.92      0.93        53
      benign       0.96      0.97      0.96        90

    accuracy                           0.95       143
   macro avg       0.95      0.95      0.95       143
weighted avg       0.95      0.95      0.95       143



### Step 8: Visualization (Accuracy Comparison & OOB Convergence)

> 💡 *Visual Diagnostics:*  
> 1. **Accuracy Bar Chart:** Visualizes the generalization gain from stabilizing 1 noisy tree into 100 bagged trees.
> 2. **Confusion Matrix:** Audits false negatives (missed malignant tumors) vs false positives.
> 3. **OOB Convergence Curve (`n_trees_range`):** Confirms where adding more trees stabilizes OOB accuracy (`~80–120` trees) so you don't waste compute training `10,000` unnecessary trees.

In [ ]:
# 8. Visualization: Accuracy Comparison, Confusion Matrix & OOB Convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy Comparison
bars = axes[0].bar(
    ["Single Decision Tree", "Bagging (100 Trees)"],
    [acc_single, acc_bag],
    color=["#e24a33", "#2ca02c"],
    edgecolor="black",
    alpha=0.85,
    width=0.5
)
axes[0].set_ylim(0.85, 1.0)
axes[0].set_title("Single Tree vs. Bagging Ensemble Accuracy", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Test Accuracy", fontsize=11)
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2.0, b.get_height() + 0.005, f"{b.get_height():.4f}", ha='center', va='bottom', fontweight='bold')

# Plot 2: Confusion Matrix
disp_bag = ConfusionMatrixDisplay(confusion_matrix=cm_bag, display_labels=cancer_bag.target_names)
disp_bag.plot(cmap="Blues", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("Bagging Ensemble: Confusion Matrix", fontsize=13, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()

# Plot 3: OOB Score vs. Number of Estimators
n_trees_range = range(20, 151, 15)
oob_scores = []
for n in n_trees_range:
    clf = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    clf.fit(X_train_bag, y_train_bag)
    oob_scores.append(clf.oob_score_)

plt.figure(figsize=(10, 4.5))
plt.plot(n_trees_range, oob_scores, marker='o', color='#1f77b4', linewidth=2, markersize=7)
plt.axhline(y=max(oob_scores), color='red', linestyle='--', label=f'Peak OOB Score ({max(oob_scores):.4f})')
plt.title("Bagging Variance Reduction: OOB Score vs. Number of Trees", fontsize=13, fontweight="bold")
plt.xlabel("Number of Estimators (Trees)", fontsize=11)
plt.ylabel("OOB Score", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()


---
# 2. Boosting (AdaBoost)

AdaBoost (**Adaptive Boosting**) is a sequential ensemble method that converts a collection of **weak learners** (models that perform only slightly better than random guessing, typically 1-split decision stumps) into a single strong learner by sequentially focusing on misclassified samples.

### 📐 Core Mathematical Mechanics: Sample Weight Updating
1. Initialize sample weights uniformly: $w_i^{(1)} = \frac{1}{N}$.
2. At boosting stage $m=1, \dots, M$:
   - Train weak learner $h_m(x)$ minimizing weighted classification error $\epsilon_m = \sum_{i: y_i \ne h_m(x_i)} w_i^{(m)}$.
   - Compute learner voting power (importance weight):
     $$\alpha_m = \frac{1}{2} \ln\left( \frac{1 - \epsilon_m}{\epsilon_m} \right)$$
     *(Learners with error near 0 receive massive positive votes; learners with error near 50% receive 0 votes).*
   - Update sample weights for the next learner:
     $$w_i^{(m+1)} = w_i^{(m)} \exp\left( -\alpha_m y_i h_m(x_i) \right)$$
     *(Correctly classified samples are down-weighted; misclassified samples are multiplied by $e^{\alpha_m}$ to force the next tree to focus on them!).*
3. Final ensemble decision is a weighted sign aggregation: $H(x) = \text{sign}\left( \sum_{m=1}^M \alpha_m h_m(x) \right)$.

> 🍋 **Layman's Analogy:**  
> Imagine a student preparing for an exam with 100 practice flashcards. On round 1, they take the test and miss 20 cards. On round 2, the tutor puts heavy red highlighter on those 20 missed cards so the student spends 80% of their study time mastering their specific weaknesses. Each round focuses aggressively on the mistakes made by the previous round until no mistakes remain!

### 🌍 Real-World Use Cases:
- **Lightweight Real-Time Edge Detection:** Historically powered the famous Viola–Jones face detection cascade because evaluating shallow stumps (`max_depth=1`) takes microseconds.
- **Clean Low-Noise Binary Tabular Classification:** Rapidly turning simple single-feature threshold rules (e.g., `worst radius <= 16.8`) into strong additive classifiers.

### ⚠️ Pitfalls of AdaBoost:
- **Extreme Sensitivity to Outliers & Mislabeled Data:** Because AdaBoost exponentially increases sample weights $\exp(\alpha_m)$ on misclassified points, noisy or mislabeled patient records quickly hijack the entire ensemble!
- **Strictly Sequential Training:** Tree $m$ cannot start until tree $m-1$ computes sample weights $w_i^{(m)}$, so AdaBoost cannot parallelize across trees (`n_jobs` doesn't speed up boosting rounds).
- **Requires Learning Rate Tuning:** Setting `learning_rate` too high (`> 1.0`) with too many stumps causes oscillation or overfitting on borderline cases.

### Step 1: Importing (AdaBoost)

> 💡 *Beginner Tip:* We import `AdaBoostClassifier` from `sklearn.ensemble` alongside `DecisionTreeClassifier` configured as a 1-split decision stump.

In [10]:
# 1. Importing libraries for AdaBoost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



### Step 2: Loading Dataset

> 💡 *Real-World Context:* Re-using the Breast Cancer feature table to evaluate sequential bias reduction across weak learners.

In [11]:
# 2. Loading Dataset
cancer_ada = load_breast_cancer(as_frame=True)
X_ada = cancer_ada.data
y_ada = cancer_ada.target
display(X_ada.head(2))


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902


### Step 3: Split (Stratified Partitioning)

> ⚠️ *Critical Engineering Concept:* Partitioning $25\%$ unseen test holdout with identical random seed (`random_state=42`) for rigorous model comparison.

In [14]:
# 3. Train/Test Split
X_train_ada, X_test_ada, y_train_ada, y_test_ada = train_test_split(
    X_ada, y_ada, test_size=0.25, random_state=42, stratify=y_ada
)
print(f"Training samples: {len(X_train_ada)} | Testing samples: {len(X_test_ada)}")


Training samples: 426 | Testing samples: 143


### Step 4: Train (AdaBoost & Decision Stumps Setup)

> 💡 *Hyperparameter Intuition — Why `max_depth=1`, `n_estimators=100`, & `learning_rate=0.5` Are Used:*  
> - **`DecisionTreeClassifier(max_depth=1)` (Decision Stump):** A 1-split tree uses only ONE feature threshold (e.g., `worst concave points <= 0.14`)—high bias, ultra-low variance.
> - **`learning_rate=0.5`:** Shrinks each stump's weight $\alpha_m$ by half across `100` sequential boosting rounds, preventing early stumps from overreacting to borderline breast cancer biopsies.

In [15]:
# 4. Train: Model Initialization
stump = DecisionTreeClassifier(max_depth=1, random_state=42)
adaboost = AdaBoostClassifier(
    estimator=stump,
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)
print("AdaBoost model initialized:", adaboost)


AdaBoost model initialized: AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,
                                                    random_state=42),
                   learning_rate=0.5, n_estimators=100, random_state=42)


### Step 5: Fit (Sequential Sample Re-weighting)

> 💡 *Mathematical Insight:* During `.fit()`, AdaBoost trains weak learners sequentially, multiplying the weights of misclassified samples by $\exp(\alpha_m)$ so subsequent stumps focus directly on previous errors.

In [16]:
# 5. Fit: Model Training
adaboost.fit(X_train_ada, y_train_ada)
print("AdaBoost training complete across 100 boosting stages!")


AdaBoost training complete across 100 boosting stages!


### Step 6: Predict (Weighted Sign Voting)

> 💡 *Beginner Tip:* Individual stumps cast votes weighted by their diagnostic competence $\alpha_m = \frac{1}{2} \ln \frac{1-\epsilon_m}{\epsilon_m}$.

In [17]:
# 6. Predict: Model Inference
y_pred_ada = adaboost.predict(X_test_ada)
print("First 10 AdaBoost Predictions:", y_pred_ada[:10])


First 10 AdaBoost Predictions: [1 0 1 1 1 0 0 0 0 1]


### Step 7: Quantitative Evaluation (AdaBoost Scorecard)

> 💡 *How to Interpret Ensemble Metrics:*  
> Notice how combining `100` tiny 1-split decision stumps — each barely checking one tumor metric — yields $>95\%$ test accuracy via sequential bias reduction!

In [18]:
# 7. Evaluation
acc_ada = accuracy_score(y_test_ada, y_pred_ada)
cm_ada = confusion_matrix(y_test_ada, y_pred_ada)

print("=" * 60)
print(f"ADABOOST ACCURACY SCORE: {acc_ada:.4f} ({acc_ada*100:.2f}%)")
print("=" * 60)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test_ada, y_pred_ada, target_names=cancer_ada.target_names))


ADABOOST ACCURACY SCORE: 0.9650 (96.50%)
CLASSIFICATION REPORT:
              precision    recall  f1-score   support

   malignant       0.98      0.92      0.95        53
      benign       0.96      0.99      0.97        90

    accuracy                           0.97       143
   macro avg       0.97      0.96      0.96       143
weighted avg       0.97      0.97      0.96       143



### Step 8: Visualization (Staged Accuracy Progression & Confusion Matrix)

> 💡 *Visual Diagnostics — Why `adaboost.staged_score()` Is Used:*  
> `staged_score(X, y)` yields the cumulative ensemble accuracy after *every single boosting round* (`1..100`). Comparing `staged_train` vs `staged_test` immediately shows whether training accuracy keeps climbing toward `1.00` while test accuracy plateaus or drops (indicating overfitting).

In [ ]:
# 8. Visualization: Staged Accuracy Progression & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Staged Score Progression
staged_train = list(adaboost.staged_score(X_train_ada, y_train_ada))
staged_test = list(adaboost.staged_score(X_test_ada, y_test_ada))

axes[0].plot(range(1, 101), staged_train, label="Training Accuracy", color="#1f77b4", linewidth=2)
axes[0].plot(range(1, 101), staged_test, label="Testing Accuracy", color="#d95f02", linewidth=2)
axes[0].set_title("AdaBoost Learning Curve (Staged Accuracy)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Boosting Rounds", fontsize=11)
axes[0].set_ylabel("Accuracy", fontsize=11)
axes[0].legend()

# Plot 2: Confusion Matrix
disp_ada = ConfusionMatrixDisplay(confusion_matrix=cm_ada, display_labels=cancer_ada.target_names)
disp_ada.plot(cmap="Oranges", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("AdaBoost: Confusion Matrix", fontsize=13, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()


---
# 3. Stacking (Stacked Generalization)

Stacking combines predictions from multiple **heterogeneous model families** (e.g., Random Forest, K-Nearest Neighbors, Naïve Bayes, and Logistic Regression) using a secondary meta-model (typically Logistic Regression) trained to learn which base model to trust for each type of sample.

### 🔬 Deep Dive: Preventing Meta-Model Data Leakage via Out-of-Fold Predictions
If base models predict directly on their own training data, their predictions will be overly optimistic (overfitted). If the meta-model trains on these biased predictions, it will overfit catastrophically!  
`StackingClassifier` uses internal $K$-fold cross-validation: for each fold, base models are trained on $K-1$ splits and generate predictions on the holdout fold. The meta-model is trained strictly on these out-of-fold predictions to ensure genuine generalization.

> 🍋 **Layman's Analogy:**  
> Think of a hospital board reviewing a difficult patient. You consult three distinct specialists: a seasoned radiologist (Random Forest), an experienced geneticist (KNN), and an oncologist (Naïve Bayes). Instead of having them vote equally, a chief medical officer (the Meta-Learner) listens to all three opinions and decides who is most credible for this specific tumor type!

### 🌍 Real-World Use Cases:
- **Kaggle & Competitive ML Leaderboards:** Combining fundamentally distinct inductive biases (tree ensembles + neural networks + linear/distance models) for final $0.3\%$–$1.0\%$ generalization gains.
- **Multi-Modal Decision Systems:** Blending tabular risk scores, NLP text classifiers, and computer vision confidence outputs via a unified calibrated meta-classifier.

### ⚠️ Pitfalls of Stacking:
- **Meta-Feature Data Leakage Without Cross-Validation:** Training the meta-learner on in-sample training predictions (rather than `cv=5` out-of-fold predictions) tricks the meta-learner into trusting whichever base model memorized the training data most (e.g., unpruned RF)!
- **No Gain from Redundant Base Models:** Stacking 3 similar Random Forests wastes compute; errors across base estimators must be **diverse / weakly correlated**.
- **High Serving Latency & Complexity:** Deploying a stacked pipeline in production requires executing and maintaining all 3 base models + the meta-model for every inference request.

### Step 1: Importing (Stacking)

> 💡 *Beginner Tip:* We import `StackingClassifier`, heterogeneous base models (`RandomForestClassifier`, `KNeighborsClassifier`, `GaussianNB`), and the meta-estimator (`LogisticRegression`).

In [ ]:
# 1. Importing libraries for Stacking
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



### Step 2: Loading Dataset

> 💡 *Real-World Context:* We evaluate whether blending diverse algorithmic families (tree, distance-based, and probabilistic) outperforms any single paradigm alone.

In [ ]:
# 2. Loading Dataset
cancer_stack = load_breast_cancer(as_frame=True)
X_stack = cancer_stack.data
y_stack = cancer_stack.target
display(X_stack.head(2))


### Step 3: Split (Stratified Partitioning)

> ⚠️ *Critical Engineering Concept:* Stratified 75/25 train/test partition preserving class balance across folds.

In [ ]:
# 3. Train/Test Split
X_train_stack, X_test_stack, y_train_stack, y_test_stack = train_test_split(
    X_stack, y_stack, test_size=0.25, random_state=42, stratify=y_stack
)
print(f"Training samples: {len(X_train_stack)} | Testing samples: {len(X_test_stack)}")


### Step 4: Train (Heterogeneous Base Architecture & Meta-Learner)

> 💡 *Hyperparameter Intuition — Why `make_pipeline(StandardScaler(), KNeighborsClassifier(5))` Is Used ONLY for KNN:*  
> Look closely at our 3 base estimators operating on `breast_cancer` (`569 x 30`):
> 1. **`rf` (`RandomForestClassifier`):** Scale-invariant tree splits—works directly on unscaled `worst area` (`185.2–4254.0`).
> 2. **`knn` (`make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`):** KNN relies on Euclidean distance! On unscaled `breast_cancer`, `worst area` (`std = 569.4`) would completely drown out `smoothness error` (`std = 0.003`), so we **must** scale before KNN. Wrapping `StandardScaler()` inside `make_pipeline()` ensures `StandardScaler` fits cleanly *inside each of the `cv=5` internal folds*—zero validation leakage!
> 3. **`nb` (`GaussianNB`):** Computes independent per-feature Gaussians, needing no scaling.
> - **`final_estimator=LogisticRegression()`:** Learns linear trust weights over the 3 base models' probability outputs.

In [ ]:
# 4. Train: Model Initialization
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('knn', make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))),
    ('nb', GaussianNB())
]

final_meta_learner = LogisticRegression(random_state=42)

stack_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=final_meta_learner,
    cv=5,
    n_jobs=-1
)
print("Stacking Classifier initialized with 3 heterogeneous base estimators.")


### Step 5: Fit (Out-of-Fold Meta-Training)

> 💡 *Mathematical Insight:*  
> `stack_clf.fit(X_train_stack, y_train_stack)` executes 5-fold cross-validation for every base estimator to create unbiased `(426, 3)` out-of-fold probability predictions, trains `LogisticRegression` on those out-of-fold scores, and then refits all 3 base estimators on the full 426 training samples for test inference.

In [ ]:
# 5. Fit: Model Training
stack_clf.fit(X_train_stack, y_train_stack)

# Also fit base models individually on test set to compare performance
base_scores = {}
for name, estimator in base_estimators:
    estimator.fit(X_train_stack, y_train_stack)
    base_scores[name] = accuracy_score(y_test_stack, estimator.predict(X_test_stack))

print("Stacking ensemble trained successfully!")


### Step 6: Predict (Meta-Learner Inference)

> 💡 *Beginner Tip:* During inference, test samples pass through all 3 base models, and their predictions are fed as synthetic features into the meta-estimator to produce the final classification.

In [ ]:
# 6. Predict: Model Inference
y_pred_stack = stack_clf.predict(X_test_stack)
print("First 10 Stacking Predictions:", y_pred_stack[:10])


### Step 7: Quantitative Evaluation (Stacking vs. Base Specialists)

> 💡 *How to Interpret Ensemble Metrics:*  
> Compare the individual test accuracies of `Random Forest`, `Scaled KNN`, and `Gaussian NB` against `Stacking Ensemble` to verify that blending diverse specialists matches or surpasses the strongest individual model.

In [ ]:
# 7. Evaluation
acc_stack = accuracy_score(y_test_stack, y_pred_stack)
cm_stack = confusion_matrix(y_test_stack, y_pred_stack)

print("=" * 60)
print("STACKING CLASSIFIER EVALUATION REPORT")
print("=" * 60)
print(f"Base Model 1 (Random Forest) Accuracy: {base_scores['rf']:.4f}")
print(f"Base Model 2 (Scaled KNN) Accuracy:    {base_scores['knn']:.4f}")
print(f"Base Model 3 (Gaussian NB) Accuracy:   {base_scores['nb']:.4f}")
print(f"Stacking Meta-Model Accuracy:          {acc_stack:.4f} ({acc_stack*100:.2f}%)")
print("=" * 60)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test_stack, y_pred_stack, target_names=cancer_stack.target_names))


### Step 8: Visualization (Specialists vs. Meta-Model Performance)

> 💡 *Visual Diagnostics:*  
> Side-by-side comparison showing how each specialist performs on the 143 test biopsies relative to the meta-learner, paired with the final confusion matrix.

In [ ]:
# 8. Visualization: Base Models vs Meta-Model & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Model Comparison Bar Chart
model_names = ['Random Forest', 'KNN (Scaled)', 'Gaussian NB', 'Stacking Ensemble']
accuracies = [base_scores['rf'], base_scores['knn'], base_scores['nb'], acc_stack]
colors = ['#7570b3', '#1b9e77', '#d95f02', '#2ca02c']

bars = axes[0].bar(model_names, accuracies, color=colors, edgecolor='black', alpha=0.85, width=0.55)
axes[0].set_ylim(0.85, 1.0)
axes[0].set_title("Base Estimators vs. Stacking Meta-Learner", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Test Accuracy", fontsize=11)
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2.0, b.get_height() + 0.004, f"{b.get_height():.4f}", ha='center', va='bottom', fontweight='bold')

# Plot 2: Confusion Matrix
disp_stack = ConfusionMatrixDisplay(confusion_matrix=cm_stack, display_labels=cancer_stack.target_names)
disp_stack.plot(cmap="Purples", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("Stacking Ensemble: Confusion Matrix", fontsize=13, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()


---
# 4. Random Forest

Random Forest extends Bagging by introducing **feature randomness (the Random Subspace Method)**:
- In standard Bagging, if one or two features are overwhelmingly dominant (e.g., `worst perimeter` in tumor data), almost every bootstrap tree will pick that same feature for its root split, making the trees highly correlated ($\rho \gg 0$).
- Random Forest forces each tree to choose its best split from only a random candidate subset of $m = \sqrt{p}$ features.
- This dramatically **de-correlates the trees**, driving pairwise correlation $\rho \to 0$ and maximizing the variance reduction benefit of Bagging!

> 🍋 **Layman's Analogy:**  
> Imagine 100 detectives solving a mystery. If you give all 100 detectives all the clues, they will all focus on the obvious bloodstain on the rug and come to identical conclusions. But if you randomly assign each detective only 3 specific clues (one gets the phone records, another gets the footprint, another gets the bank statement), each detective uncovers a unique perspective. When they vote, their collective verdict is bulletproof!

### 🌍 Real-World Use Cases:
- **Gold-Standard Tabular Baseline:** First-choice production baseline for fraud detection, medical diagnosis, customer churn, and credit scoring because it requires zero feature scaling and tolerates outliers/noisy columns gracefully.
- **High-Dimensional Feature Selection (`feature_importances_`):** Ranking thousands of genomic SNPs or clinical measurements to identify top biomarkers.

### ⚠️ Pitfalls of Random Forest:
- **Biased Gini MDI Feature Importances:** Default `feature_importances_` favors high-cardinality continuous features (`worst area`) over discrete/low-cardinality predictors. Verify clinical rankings with `permutation_importance` or SHAP!
- **Poor Extrapolation Outside Training Bounds:** Because Random Forest averages leaf targets, it can never extrapolate beyond the minimum/maximum values observed during training (critical if used for regression).
- **Large Model Memory Size:** Storing hundreds of deep unpruned trees can bloat serialized `.pkl` / `.joblib` files to tens or hundreds of megabytes on million-row datasets.

### Step 1: Importing (Random Forest)

> 💡 *Beginner Tip:* We import `RandomForestClassifier` from `sklearn.ensemble` alongside diagnostic visualization tools.

In [ ]:
# 1. Importing libraries for Random Forest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



### Step 2: Loading Dataset

> 💡 *Real-World Context:* Standard Wisconsin Breast Cancer benchmark.

In [ ]:
# 2. Loading Dataset
cancer_rf = load_breast_cancer(as_frame=True)
X_rf = cancer_rf.data
y_rf = cancer_rf.target
display(X_rf.head(2))


### Step 3: Split (Stratified Partitioning)

> ⚠️ *Critical Engineering Concept:* Consistent stratified splitting for benchmark comparability.

In [ ]:
# 3. Train/Test Split
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_rf, y_rf, test_size=0.25, random_state=42, stratify=y_rf
)
print(f"Training samples: {len(X_train_rf)} | Testing samples: {len(X_test_rf)}")


### Step 4: Train (De-correlated Forest Setup)

> 💡 *Hyperparameter Intuition — Why `max_features='sqrt'` & `oob_score=True` Are Used:*  
> - **`max_features='sqrt'`:** With `30` features in `breast_cancer`, each tree node evaluates only $\lfloor\sqrt{30}\rfloor = 5$ random candidate features per split instead of all 30! This prevents `worst concave points` or `worst perimeter` from monopolizing every root split across all 100 trees.
> - **`n_estimators=100`, `oob_score=True`, `n_jobs=-1`:** Grows 100 de-correlated trees across all CPU cores with built-in out-of-bag validation.

In [ ]:
# 4. Train: Model Initialization
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
print("Random Forest model initialized:", rf_clf)


### Step 5: Fit (Parallel De-correlated Tree Training)

> 💡 *Mathematical Insight:* Feature subsampling forces individual trees to uncover diverse diagnostic signals, driving pairwise tree correlation $\rho \to 0$ and maximizing ensemble variance reduction.

In [ ]:
# 5. Fit: Model Training
rf_clf.fit(X_train_rf, y_train_rf)
print(f"Random Forest fitted! Out-of-Bag (OOB) Score: {rf_clf.oob_score_:.4f}")


### Step 6: Predict (Majority Voting Inference)

> 💡 *Beginner Tip:* 100 de-correlated trees vote independently, yielding a highly stable, low-variance majority classification.

In [ ]:
# 6. Predict: Model Inference
y_pred_rf = rf_clf.predict(X_test_rf)
print("First 10 Random Forest Predictions:", y_pred_rf[:10])


### Step 7: Quantitative Evaluation (Random Forest Scorecard)

> 💡 *How to Interpret Ensemble Metrics:*  
> Inspect test accuracy, OOB accuracy (`rf_clf.oob_score_`), and malignant recall—notice how feature de-correlation typically outperforms plain Bagging from Section 1.

In [ ]:
# 7. Evaluation
acc_rf = accuracy_score(y_test_rf, y_pred_rf)
cm_rf = confusion_matrix(y_test_rf, y_pred_rf)

print("=" * 60)
print("RANDOM FOREST EVALUATION REPORT")
print("=" * 60)
print(f"Overall Test Accuracy:  {acc_rf:.4f} ({acc_rf*100:.2f}%)")
print(f"Out-of-Bag (OOB) Score: {rf_clf.oob_score_:.4f} ({rf_clf.oob_score_*100:.2f}%)")
print("=" * 60)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test_rf, y_pred_rf, target_names=cancer_rf.target_names))


### Step 8: Visualization (Feature Importances & OOB Error Curve)

> 💡 *Visual Diagnostics:*  
> 1. **Top 15 Feature Importances (`feature_importances_`):** Reveals which nuclear traits (`worst perimeter`, `worst concave points`, `worst radius`) carry the highest Mean Decrease in Gini Impurity across the 100 trees.
> 2. **OOB Error Curve (`1.0 - rf_temp.oob_score_`):** Plotting OOB *error rate* across `20..200` trees confirms error settles smoothly as forest size reaches `~100` trees.

In [ ]:
# 8. Visualization: Feature Importances, Confusion Matrix & OOB Error Curve
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Feature Importances
rf_importances = pd.Series(rf_clf.feature_importances_, index=cancer_rf.feature_names).sort_values(ascending=False).head(15)
rf_importances.sort_values().plot(kind="barh", ax=axes[0], color="#2ca02c", edgecolor="black", alpha=0.85)
axes[0].set_title("Random Forest: Top 15 Feature Importances (Gini MDI)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Mean Decrease in Impurity", fontsize=11)

# Plot 2: Confusion Matrix
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=cancer_rf.target_names)
disp_rf.plot(cmap="Greens", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("Random Forest: Confusion Matrix", fontsize=13, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()

# Plot 3: OOB Error vs. Number of Trees
tree_counts = range(20, 201, 20)
oob_errors = []
for n in tree_counts:
    rf_temp = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf_temp.fit(X_train_rf, y_train_rf)
    oob_errors.append(1.0 - rf_temp.oob_score_)

plt.figure(figsize=(10, 4.5))
plt.plot(tree_counts, oob_errors, marker='s', color='#d95f02', linewidth=2, markersize=7)
plt.title("Random Forest: OOB Error Rate vs. Number of Trees", fontsize=13, fontweight="bold")
plt.xlabel("Number of Trees", fontsize=11)
plt.ylabel("OOB Error Rate (1 - OOB Score)", fontsize=11)
plt.tight_layout()
plt.show()


---
# 5. Gradient Boosting (GBDT)

Gradient Boosted Decision Trees (GBDT) formulate boosting as **Gradient Descent in Function Space**:
Instead of re-weighting misclassified samples like AdaBoost, GBDT trains each subsequent tree directly on the **pseudo-residuals** (the negative gradient of the differentiable loss function with respect to current predictions):
$$r_{im} = - \left[ \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \right]_{F(x) = F_{m-1}(x)}$$
The new tree $h_m(x)$ is fitted to predict these residuals, and updated with a shrinkage rate $\eta$:
$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

> 🍋 **Layman's Analogy:**  
> Imagine shooting arrows at a bullseye. Your first shot hits 10 cm to the right (residual = +10 cm). Instead of firing an entirely new blind shot, you aim your second arrow exactly 10 cm to the left! If that shot still misses by 2 cm high, your third arrow adjusts 2 cm down. Each shot corrects the precise geometric error left behind by all previous shots!

### 🌍 Real-World Use Cases:
- **Tabular Predictive Analytics:** Search ranking, insurance pricing, customer lifetime value forecasting, and structured clinical outcome modeling where shallow sequential trees deliver top-tier accuracy.
- **Custom Differentiable Loss Tasks:** Optimizing quantile regression (predicting 90th-percentile delivery time bounds) or Huber loss natively.

### ⚠️ Pitfalls of Gradient Boosting (GBDT):
- **Overfitting With Too Many Trees / Deep Trees:** Unlike Random Forest (where adding more trees cannot overfit), adding too many GBDT stages causes the model to chase noisy training residuals while test log-loss starts climbing! Always monitor validation deviance (`staged_predict_proba`).
- **Slow Sequential CPU Training in Scikit-Learn:** Scikit-Learn's `GradientBoostingClassifier` runs single-threaded exact splits and lacks GPU acceleration (for $>50{,}000$ rows, switch to `HistGradientBoostingClassifier`, XGBoost, LightGBM, or CatBoost).
- **Sensitivity to `learning_rate` vs `n_estimators` Coupling:** Halving `learning_rate` generally requires doubling `n_estimators` to converge.

### Step 1: Importing (Gradient Boosting)

> 💡 *Beginner Tip:* We import `GradientBoostingClassifier` from `sklearn.ensemble`.

In [ ]:
# 1. Importing libraries for Gradient Boosting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



### Step 2: Loading Dataset

> 💡 *Real-World Context:* Standard Breast Cancer dataset.

In [ ]:
# 2. Loading Dataset
cancer_gb = load_breast_cancer(as_frame=True)
X_gb = cancer_gb.data
y_gb = cancer_gb.target
display(X_gb.head(2))


### Step 3: Split (Stratified Partitioning)

> ⚠️ *Critical Engineering Concept:* Stratified 75/25 split.

In [ ]:
# 3. Train/Test Split
X_train_gb, X_test_gb, y_train_gb, y_test_gb = train_test_split(
    X_gb, y_gb, test_size=0.25, random_state=42, stratify=y_gb
)
print(f"Training samples: {len(X_train_gb)} | Testing samples: {len(X_test_gb)}")


### Step 4: Train (Gradient Boosting Configuration)

> 💡 *Hyperparameter Intuition — Why `max_depth=3`, `learning_rate=0.1`, & `subsample=0.8` Are Used:*  
> - **`max_depth=3`:** Unlike Random Forest's deep trees, boosting prefers shallow weak learners (`depth=3`, max 8 leaves) that capture 3-way feature interactions without memorizing single patients.
> - **`learning_rate=0.1`:** Shrinks each residual tree's step size so subsequent trees refine predictions gradually.
> - **`subsample=0.8` (Stochastic Gradient Boosting):** Each tree trains on a random $80\%$ subsample of the 426 training biopsies, injecting randomness that lowers variance and unlocks out-of-bag regularization.

In [ ]:
# 4. Train: Model Initialization
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)
print("Gradient Boosting model initialized:", gb_clf)


### Step 5: Fit (Iterative Pseudo-Residual Fitting)

> 💡 *Mathematical Insight:* GBDT computes negative gradients of the binary deviance loss and trains each subsequent tree to predict these residuals, taking small steps of size $\eta=0.1$.

In [ ]:
# 5. Fit: Model Training
gb_clf.fit(X_train_gb, y_train_gb)
print("Gradient Boosting training complete!")


### Step 6: Predict (Ensemble Score Inference)

> 💡 *Beginner Tip:* Predictions accumulate across all 100 boosting stages: $F(x) = F_0 + \sum_{m=1}^{100} \eta h_m(x)$.

In [ ]:
# 6. Predict: Model Inference
y_pred_gb = gb_clf.predict(X_test_gb)
print("First 10 GBDT Predictions:", y_pred_gb[:10])


### Step 7: Quantitative Evaluation (GBDT Scorecard)

> 💡 *How to Interpret Ensemble Metrics:*  
> Evaluate test accuracy, malignant recall, and F1 score to see how gradient-driven bias reduction compares against parallel bagging/Random Forest.

In [ ]:
# 7. Evaluation
acc_gb = accuracy_score(y_test_gb, y_pred_gb)
cm_gb = confusion_matrix(y_test_gb, y_pred_gb)

print("=" * 60)
print(f"GRADIENT BOOSTING ACCURACY: {acc_gb:.4f} ({acc_gb*100:.2f}%)")
print("=" * 60)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test_gb, y_pred_gb, target_names=cancer_gb.target_names))


### Step 8: Visualization (Deviance Curves & Confusion Matrix)

> 💡 *Visual Diagnostics — Why `staged_predict_proba()` + `log_loss()` vs `train_score_` Are Plotted:*  
> `gb_clf.train_score_` tracks training deviance at every iteration `1..100`, while `[log_loss(y_test_gb, p) for p in gb_clf.staged_predict_proba(X_test_gb)]` measures test deviance at each round! If the red test curve bends upward while the blue training curve keeps dropping, you have pinpointed the exact iteration where overfitting begins.

In [ ]:
# 8. Visualization: Deviance Curves & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Deviance Curve
test_deviance = [log_loss(y_test_gb, prob) for prob in gb_clf.staged_predict_proba(X_test_gb)]
train_deviance = gb_clf.train_score_

axes[0].plot(range(1, 101), train_deviance, label="Training Deviance", color="#1f77b4", linewidth=2)
axes[0].plot(range(1, 101), test_deviance, label="Testing Deviance", color="#e24a33", linewidth=2)
axes[0].set_title("Gradient Boosting Deviance (Log-Loss) per Iteration", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Boosting Iterations", fontsize=11)
axes[0].set_ylabel("Log-Loss Deviance", fontsize=11)
axes[0].legend()

# Plot 2: Confusion Matrix
disp_gb = ConfusionMatrixDisplay(confusion_matrix=cm_gb, display_labels=cancer_gb.target_names)
disp_gb.plot(cmap="Blues", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("Gradient Boosting: Confusion Matrix", fontsize=13, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()


---
# 6. CatBoost (Categorical Boosting)

## 6.1 Baseline Benchmark: Oblivious Trees on Numerical Data (Breast Cancer)

CatBoost (developed by Yandex) is a state-of-the-art gradient boosting framework designed to solve two core challenges in traditional GBDT:
1. **Symmetric (Oblivious) Trees:** Uses identical split conditions across entire levels of the tree. This acts as strong structural regularization against overfitting and enables lightning-fast model evaluation compiled to CPU SIMD/bit-vector operations.
2. **Ordered Boosting:** Traditional GBDT uses the same dataset to compute residuals and train base models, causing target leakage and statistical bias. CatBoost uses ordered permutations to calculate unbiased online gradients.

> 🍋 **Layman's Analogy:**  
> Traditional decision trees are crooked: left branches might grow 6 levels deep while right branches stop at 2. CatBoost builds perfectly symmetrical balance scales: every branch on the left matches every branch on the right. This symmetry makes it impossible for the tree to memorize weird noisy corners of the data, and lets modern computer processors evaluate predictions at extreme speed!

### 🌍 Real-World Use Cases:
- **Mixed Tabular & High-Cardinality Business Data:** E-commerce conversion, telecom churn, ad click-through prediction, and banking risk where IDs, merchant codes, and categorical strings abound.
- **Low-Latency CPU Serving:** Oblivious trees evaluate via flat lookup index tables without pointer branching, running dramatically faster in production web microservices.

### ⚠️ Pitfalls of CatBoost:
- **Automatic `catboost_info/` Workspace Clutter & Slow Default Training on Pure Numerics:** Without `verbose=0` or `train_dir`, CatBoost writes diagnostic logs to disk; and on purely numerical datasets without categoricals, XGBoost/LightGBM can match accuracy with faster tuning sweeps.
- **Passing Pre-One-Hot-Encoded Columns:** One-hot encoding 50-level categories *before* CatBoost strips away CatBoost's core advantage (`Ordered Target Statistics`) and fragments symmetrical splits.

### Step 1: Importing (CatBoost)

> 💡 *Beginner Tip:* We import `CatBoostClassifier` from the high-performance `catboost` library.

In [ ]:
# 1. Importing libraries for CatBoost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



### Step 2: Loading Dataset

> 💡 *Real-World Context:* Standard Breast Cancer benchmark.

In [ ]:
# 2. Loading Dataset
cancer_cb = load_breast_cancer(as_frame=True)
X_cb = cancer_cb.data
y_cb = cancer_cb.target
display(X_cb.head(2))


### Step 3: Split (Stratified Partitioning)

> ⚠️ *Critical Engineering Concept:* Stratified partitioning.

In [ ]:
# 3. Train/Test Split
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X_cb, y_cb, test_size=0.25, random_state=42, stratify=y_cb
)
print(f"Training samples: {len(X_train_cb)} | Testing samples: {len(X_test_cb)}")


### Step 4: Train (CatBoost Symmetrical Tree Setup)

> 💡 *Hyperparameter Intuition:* `depth=4` creates symmetric oblivious trees evaluated via fast bitwise CPU masks, and `iterations=100` controls boosting rounds.

In [ ]:
# 4. Train: Model Initialization
cat_clf = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    eval_metric='Logloss',
    verbose=0,
    random_state=42
)
print("CatBoost model initialized:", cat_clf)


### Step 5: Fit (Ordered Boosting on Evaluation Set)

> 💡 *Mathematical Insight:* CatBoost executes ordered boosting with dynamic validation loss tracking (`eval_metric='Logloss'`) to prevent prediction shift.

In [ ]:
# 5. Fit: Model Training
cat_clf.fit(X_train_cb, y_train_cb, eval_set=(X_test_cb, y_test_cb), verbose=False)
print("CatBoost model training complete!")


### Step 6: Predict (Fast Symmetrical Tree Inference)

> 💡 *Beginner Tip:* Oblivious trees evaluate predictions at blazing speed because all tree leaves at depth $d$ share identical binary split indices.

In [ ]:
# 6. Predict: Model Inference
y_pred_cb = cat_clf.predict(X_test_cb)
print("First 10 CatBoost Predictions:", y_pred_cb[:10])


### Step 7: Quantitative Evaluation (CatBoost Scorecard)

> 💡 *How to Interpret Ensemble Metrics:*  
> Assess CatBoost's test accuracy and precision/recall balance on the `143` holdout Breast Cancer biopsies.

In [ ]:
# 7. Evaluation
acc_cb = accuracy_score(y_test_cb, y_pred_cb)
cm_cb = confusion_matrix(y_test_cb, y_pred_cb)

print("=" * 60)
print(f"CATBOOST ACCURACY SCORE: {acc_cb:.4f} ({acc_cb*100:.2f}%)")
print("=" * 60)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test_cb, y_pred_cb, target_names=cancer_cb.target_names))


### Step 8: Visualization (Feature Importances & Confusion Matrix)

> 💡 *Visual Diagnostics:*  
> `cat_clf.get_feature_importance()` computes PredictionValuesChange—how much predictions shift on average when each feature value changes—paired with the test confusion matrix.

In [ ]:
# 8. Visualization: Feature Importances & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Feature Importances
cb_importances = pd.Series(cat_clf.get_feature_importance(), index=cancer_cb.feature_names).sort_values(ascending=False).head(15)
cb_importances.sort_values().plot(kind="barh", ax=axes[0], color="#e7298a", edgecolor="black", alpha=0.85)
axes[0].set_title("CatBoost: Top 15 Feature Importances", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Feature Importance (Prediction Value Change)", fontsize=11)

# Plot 2: Confusion Matrix
disp_cb = ConfusionMatrixDisplay(confusion_matrix=cm_cb, display_labels=cancer_cb.target_names)
disp_cb.plot(cmap="RdPu", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("CatBoost: Confusion Matrix", fontsize=13, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()


---
## 6.2 Industry Focus: Native Categorical Feature Handling in CatBoost

While Section 6.1 demonstrated CatBoost on standard continuous numerical measurements (Breast Cancer), CatBoost's true superpower in industrial machine learning is its **native handling of categorical features**.

In real-world business datasets — such as customer churn, fraud detection, credit risk, and e-commerce recommendations — features are rarely pure numbers. They are strings, categories, and IDs (e.g. `ContractType = 'Month-to-Month'`, `PaymentMethod = 'Electronic Check'`).

---

### 🔬 Three Strategies for Categorical Features in Tree Ensembles:

| Strategy | Mechanism | Key Limitations | Best Applied When |
| :--- | :--- | :--- | :--- |
| **1. One-Hot Encoding (OHE)** | Creates $K$ binary indicator columns for a feature with $K$ categories. | **Dimensionality Explosion:** For a categorical column with 100 levels (e.g., Zip Code), adds 100 sparse columns. Tree splits on single levels isolate tiny sample fractions, causing severe fragmentation and slow convergence. | Low-cardinality features ($K \le 4$) in linear models or simple shallow trees. |
| **2. Ordinal / Label Encoding** | Assigns arbitrary integers: e.g. `'Month-to-Month'=0`, `'One-Year'=1`, `'Two-Year'=2`. | **Artificial Distance Distortion:** Imposes a false linear ordering ($0 < 1 < 2$) on variables that may have no natural ordinal relationship (e.g. `'Credit Card'=0`, `'Mailed Check'=1`, `'Bank Transfer'=2`). Trees waste splits attempting to undo the artificial geometry. | Inherently ordinal variables (e.g., `'Low'=0, 'Medium'=1, 'High'=2`). |
| **3. CatBoost Native Handling (`cat_features`)** | **Ordered Target Statistics (Online Target Encoding):** Dynamically calculates smoothed target statistics using random permutations of training rows to eliminate target leakage. | Requires categorical column indices/names to be explicitly declared. | **High-cardinality and mixed tabular data** in production systems. |

---

### 📐 The Mathematics of CatBoost Ordered Target Statistics:
Instead of static Target Encoding (which causes catastrophic target leakage and overfitting), CatBoost computes target statistics over a randomly permuted historical sequence $\sigma$:
$$\hat{x}_{k}^{i} = \frac{\sum_{j=1}^{p-1} [x_{\sigma(j), k} = x_{\sigma(p), k}] \cdot y_{\sigma(j)} + a \cdot P}{\sum_{j=1}^{p-1} [x_{\sigma(j), k} = x_{\sigma(p), k}] + a}$$
Where $P$ is the global target prior, $a > 0$ is a Bayesian smoothing parameter, and only observations **prior** to sample $p$ in permutation $\sigma$ are used to compute its target statistic!


### Step 1: Importing (CatBoost & Preprocessing Encoders)

> 💡 *Beginner Tip:* We import `CatBoostClassifier`, plus `OneHotEncoder` and `OrdinalEncoder` from scikit-learn to benchmark all three categorical strategies side by side.


In [ ]:
# 1. Importing libraries for CatBoost Categorical Experimentation
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


### Step 2: Loading Dataset (Customer Subscription & Churn)

> 💡 *Real-World Context:*  
> We synthesize a representative telecommunications customer churn dataset containing:
> - **Continuous Numerical Features:** `Age`, `TenureMonths`, `MonthlyCharges`, `TotalCharges`.
> - **Categorical String Features:** `ContractType`, `PaymentMethod`, `InternetService`, `Gender`, `PaperlessBilling`.
> - **Target:** `Churn` (0 = Retained, 1 = Churned).


In [ ]:
# 2. Loading Dataset: Mixed Numerical & Categorical Customer Churn Dataset
np.random.seed(42)
n_samples = 800

contract_types    = ['Month-to-Month', 'One-Year', 'Two-Year']
payment_methods   = ['Electronic Check', 'Mailed Check', 'Bank Transfer', 'Credit Card']
internet_services = ['DSL', 'Fiber Optic', 'No']
genders           = ['Male', 'Female']
paperless         = ['Yes', 'No']

contracts      = np.random.choice(contract_types, size=n_samples, p=[0.55, 0.25, 0.20])
payments       = np.random.choice(payment_methods, size=n_samples, p=[0.35, 0.20, 0.25, 0.20])
internets      = np.random.choice(internet_services, size=n_samples, p=[0.40, 0.45, 0.15])
gender         = np.random.choice(genders, size=n_samples, p=[0.50, 0.50])
paperless_bill = np.random.choice(paperless, size=n_samples, p=[0.60, 0.40])

age             = np.random.randint(18, 75, size=n_samples)
tenure          = np.random.randint(1, 72, size=n_samples)
monthly_charges = np.round(np.random.uniform(20.0, 115.0, size=n_samples), 2)
total_charges   = np.round(tenure * monthly_charges + np.random.normal(0, 15, size=n_samples), 2)
total_charges   = np.maximum(total_charges, monthly_charges)

# Realistic churn logic driven by contract, tenure, and payment method
churn_latent = (
    0.35 * (contracts == 'Month-to-Month') +
    0.25 * (payments == 'Electronic Check') +
    0.20 * (internets == 'Fiber Optic') +
    0.15 * (paperless_bill == 'Yes') -
    0.30 * (contracts == 'Two-Year') -
    0.25 * (tenure > 36) +
    0.10 * (monthly_charges > 80)
)
churn_prob = np.clip(churn_latent + np.random.normal(0, 0.08, size=n_samples), 0.05, 0.95)
churn = (churn_prob > 0.45).astype(int)

df_churn = pd.DataFrame({
    'Age': age,
    'TenureMonths': tenure,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'ContractType': contracts,
    'PaymentMethod': payments,
    'InternetService': internets,
    'Gender': gender,
    'PaperlessBilling': paperless_bill,
    'Churn': churn
})

categorical_features = ['ContractType', 'PaymentMethod', 'InternetService', 'Gender', 'PaperlessBilling']
numerical_features   = ['Age', 'TenureMonths', 'MonthlyCharges', 'TotalCharges']

print(f"Dataset Shape: {df_churn.shape} | Churn Rate: {df_churn['Churn'].mean()*100:.1f}%")
print(f"Categorical Features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical Features ({len(numerical_features)}):   {numerical_features}")
display(df_churn.head(4))


### Step 3: Split (Stratified Partitioning Retaining Native String Types)

> ⚠️ *Critical Engineering Concept — No Manual Pre-Encoding Needed:*  
> With CatBoost, we leave categorical columns as standard Python `str` / pandas `object` types! We do **not** run one-hot encoding or manual integer mapping before splitting.


In [ ]:
# 3. Train/Test Split
X_churn = df_churn.drop(columns=['Churn'])
y_churn = df_churn['Churn']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn, y_churn, test_size=0.25, random_state=42, stratify=y_churn
)

print(f"Training set: {X_train_c.shape[0]} records | Test set: {X_test_c.shape[0]} records")
print("Data types preserved:")
print(X_train_c.dtypes)


### Step 4: Train (CatBoost Setup with `cat_features`)

> 💡 *Hyperparameter Intuition:*  
> By passing `cat_features=categorical_features`, CatBoost automatically builds internal target statistics tables, feature combinations, and one-hot thresholds during tree building without requiring external data pipelines.


In [ ]:
# 4. Train: CatBoost Initialization with cat_features
cb_categorical = CatBoostClassifier(
    iterations=150,
    learning_rate=0.08,
    depth=5,
    cat_features=categorical_features,
    eval_metric='Logloss',
    verbose=0,
    random_seed=42
)
print("CatBoost Native Classifier Initialized with cat_features:", categorical_features)


### Step 5: Fit (Ordered Boosting with On-the-Fly Category Combinations)

> 💡 *Mathematical Insight:*  
> During training, CatBoost not only encodes individual categorical features, but also evaluates **feature combinations** (e.g. `ContractType + InternetService`) on subsequent tree splits!


In [ ]:
# 5. Fit: Model Training
t0 = time.perf_counter()
cb_categorical.fit(
    X_train_c, y_train_c,
    eval_set=(X_test_c, y_test_c),
    verbose=False
)
t_fit_native = time.perf_counter() - t0
print(f"CatBoost native model trained in {t_fit_native:.3f} seconds!")


### Step 6: Predict (Model Inference on Unseen Customer Records)

> 💡 *Beginner Tip:*  
> Test inference accepts raw un-encoded pandas DataFrames directly. The compiled model applies the target statistics internally with zero Python preprocessing overhead.


In [ ]:
# 6. Predict: Model Inference
y_pred_native = cb_categorical.predict(X_test_c)
y_prob_native = cb_categorical.predict_proba(X_test_c)[:, 1]

print("First 10 Predicted Labels:  ", list(y_pred_native[:10]))
print("First 10 Churn Probabilities:", [round(p, 3) for p in y_prob_native[:10]])


### Step 7: Quantitative Evaluation (Scorecard)

> 💡 *How to Interpret Ensemble Metrics:*  
> We evaluate test accuracy, precision, recall, and F1 score for churn prediction.


In [ ]:
# 7. Evaluation
acc_native  = accuracy_score(y_test_c, y_pred_native)
prec_native = precision_score(y_test_c, y_pred_native)
rec_native  = recall_score(y_test_c, y_pred_native)
f1_native   = f1_score(y_test_c, y_pred_native)
cm_native   = confusion_matrix(y_test_c, y_pred_native)

print("=" * 60)
print(f"CATBOOST NATIVE CATEGORICAL EVALUATION REPORT")
print("=" * 60)
print(f"Accuracy:  {acc_native:.4f} ({acc_native*100:.2f}%)")
print(f"Precision: {prec_native:.4f}")
print(f"Recall:    {rec_native:.4f}")
print(f"F1 Score:  {f1_native:.4f}")
print("-" * 60)
print(classification_report(y_test_c, y_pred_native, target_names=["Retained", "Churned"]))


### Step 8: Visualization (Feature Importances with Categorical Rankings)

> 💡 *Visual Diagnostics:*  
> We inspect feature importance scores. Notice how high-cardinality categorical variables (like `ContractType` and `PaymentMethod`) rank relative to continuous metrics!


In [ ]:
# 8. Visualization: Feature Importances & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# Plot 1: Feature Importances
importances = pd.Series(cb_categorical.get_feature_importance(), index=X_churn.columns).sort_values(ascending=True)
colors = ['#3498db' if col in categorical_features else '#2ecc71' for col in importances.index]

importances.plot(kind="barh", ax=axes[0], color=colors, edgecolor="black", alpha=0.85)
axes[0].set_title("CatBoost Feature Importances (Blue=Categorical, Green=Numerical)", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Relative Importance (%)", fontsize=10)

# Plot 2: Confusion Matrix
disp_churn = ConfusionMatrixDisplay(confusion_matrix=cm_native, display_labels=["Retained", "Churned"])
disp_churn.plot(cmap="Blues", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title(f"CatBoost Confusion Matrix (Acc: {acc_native*100:.1f}%)", fontsize=11, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()


---
### 🔬 Empirical Comparison: One-Hot vs. Ordinal vs. CatBoost Native Handling

To demonstrate why CatBoost's native categorical engine is superior, we benchmark all three categorical encoding strategies on our identical `df_churn` customer dataset (`600` train / `200` test records across `5` categorical + `4` numerical columns):
1. **One-Hot Encoding (`OneHotEncoder(sparse_output=False, handle_unknown='ignore')`):**
   - **Concrete `df_churn` Example:** A customer with `ContractType='Month-to-Month'`, `PaymentMethod='Electronic Check'`, `InternetService='Fiber Optic'`, `Gender='Female'`, `PaperlessBilling='Yes'` expands from **5 string columns** into **14 binary `0/1` columns** (`[1,0,0, 0,0,1,0, 0,1,0, 1,0, 0,1]`), doubling total dimensionality from `9` $\to$ `18` features!
   - **Why `sparse_output=False` & `handle_unknown='ignore'`?** `sparse_output=False` outputs a dense NumPy array so `np.hstack` can combine it cleanly with `numerical_features` (`Age`, `TenureMonths`, `MonthlyCharges`, `TotalCharges`); `handle_unknown='ignore'` outputs all zeros `[0, 0, 0]` instead of raising an exception if an unseen category (e.g., `'Quarterly'`) arrives at inference time.
2. **Ordinal Encoding (`OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)`):**
   - **Concrete `df_churn` Example:** Maps `PaymentMethod` alphabetically to integers `'Bank Transfer' -> 0`, `'Credit Card' -> 1`, `'Electronic Check' -> 2`, `'Mailed Check' -> 3`—forcing arbitrary numerical order (`Bank < Credit < Electronic < Mailed`), while `unknown_value=-1` safely catches unseen labels.
3. **CatBoost Native (`cat_features`):** Computes unbiased Ordered Target Statistics directly on the original 9-column DataFrame with zero column explosion.

In [ ]:
# Empirical Comparison of Three Categorical Encoding Strategies
# 1. One-Hot Encoding Pipeline
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_ohe_cat = ohe.fit_transform(X_train_c[categorical_features])
X_test_ohe_cat  = ohe.transform(X_test_c[categorical_features])

X_train_ohe = np.hstack([X_train_c[numerical_features].values, X_train_ohe_cat])
X_test_ohe  = np.hstack([X_test_c[numerical_features].values, X_test_ohe_cat])

cb_ohe = CatBoostClassifier(iterations=150, learning_rate=0.08, depth=5, eval_metric='Logloss', verbose=0, random_seed=42)
t0 = time.perf_counter()
cb_ohe.fit(X_train_ohe, y_train_c, eval_set=(X_test_ohe, y_test_c), verbose=False)
t_fit_ohe = time.perf_counter() - t0
y_pred_ohe = cb_ohe.predict(X_test_ohe)
f1_ohe = f1_score(y_test_c, y_pred_ohe)
acc_ohe = accuracy_score(y_test_c, y_pred_ohe)

# 2. Ordinal Encoding Pipeline
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_oe_cat = oe.fit_transform(X_train_c[categorical_features])
X_test_oe_cat  = oe.transform(X_test_c[categorical_features])

X_train_oe = np.hstack([X_train_c[numerical_features].values, X_train_oe_cat])
X_test_oe  = np.hstack([X_test_c[numerical_features].values, X_test_oe_cat])

cb_oe = CatBoostClassifier(iterations=150, learning_rate=0.08, depth=5, eval_metric='Logloss', verbose=0, random_seed=42)
t0 = time.perf_counter()
cb_oe.fit(X_train_oe, y_train_c, eval_set=(X_test_oe, y_test_c), verbose=False)
t_fit_oe = time.perf_counter() - t0
y_pred_oe = cb_oe.predict(X_test_oe)
f1_oe = f1_score(y_test_c, y_pred_oe)
acc_oe = accuracy_score(y_test_c, y_pred_oe)

# Synthesize Strategy Comparison Table
print("=" * 80)
print(f"{'Encoding Strategy':<26} | {'Features':<10} | {'Fit Time (s)':<14} | {'Test Acc':<12} | {'Test F1':<10}")
print("-" * 80)
print(f"{'1. One-Hot Encoding':<26} | {X_train_ohe.shape[1]:<10} | {t_fit_ohe:<14.4f} | {acc_ohe*100:<11.2f}% | {f1_ohe:<10.4f}")
print(f"{'2. Ordinal Encoding':<26} | {X_train_oe.shape[1]:<10} | {t_fit_oe:<14.4f} | {acc_oe*100:<11.2f}% | {f1_oe:<10.4f}")
print(f"{'3. CatBoost Native (O.T.S.)':<26} | {X_train_c.shape[1]:<10} | {t_fit_native:<14.4f} | {acc_native*100:<11.2f}% | {f1_native:<10.4f}")
print("=" * 80)
print("""
💡 Key Architectural Insights:
1. One-Hot Encoding doubled feature dimensionality (from 9 to 18 columns), fragmenting tree splits.
2. Ordinal Encoding forced non-ordinal categorical values (Payment Methods) onto an arbitrary 1D scale.
3. CatBoost Native Handling achieved superior F1 accuracy while preserving compact input geometry!""")


---
# 7. XGBoost (Extreme Gradient Boosting)

XGBoost (Chen & Guestrin, 2016) is the industry standard for competitive tabular machine learning. Its key innovations include:
1. **2nd-Order Taylor Approximation:** While standard GBDT uses only first-order gradients ($g_i$), XGBoost approximates the loss function using both the gradient ($g_i$) and Hessian ($h_i$):
   $$\tilde{\mathcal{L}}^{(t)} \approx \sum_{i=1}^N \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$
2. **Regularized Objective:** Penalizes tree complexity with leaf count penalty $\gamma$ and L2 leaf weight penalty $\lambda$:
   $$\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^T w_j^2$$
3. **Optimal Closed-Form Leaf Weight:** For leaf node $j$ with sample subset $I_j$:
   $$w_j^* = - \frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$
4. **Exact Greedy Split Finding:** Evaluates score gain when splitting a leaf into left and right sub-leaves:
   $$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

> 🍋 **Layman's Analogy:**  
> Standard GBDT knows only which direction to walk (first derivative / gradient). XGBoost calculates both the slope of the hill AND how sharply the hill curves (second derivative / Hessian)! This curvature awareness tells XGBoost exactly how far to step without overshooting, making it vastly more stable and precise on massive datasets.

### 🌍 Real-World Use Cases:
- **Large-Scale Tabular Production Pipelines:** Real-time fraud scoring, credit underwriting, insurance claims prediction, and click-through-rate modeling on millions of rows (supports histogram splitting `tree_method='hist'` and CUDA GPUs).
- **Sparse Datasets with Missing Values (`NaN`):** XGBoost automatically learns the optimal default branch direction for `NaN` values during split finding without manual imputation!

### ⚠️ Pitfalls of XGBoost:
- **Overfitting Deep Trees Without Regularization or Early Stopping:** Raising `max_depth` (`> 8`) without tuning `min_child_weight`, `reg_lambda`, `subsample`, or `early_stopping_rounds` quickly memorizes training noise.
- **Validation Leakage if Tuning on Test Set:** Passing `(X_test, y_test)` into `eval_set` for visual curve inspection is fine for diagnostics, but in real model tuning you must reserve a *separate* validation fold (`X_val`) for `early_stopping_rounds`!
- **Categorical String Handling:** Prior to enabling `enable_categorical=True` (with pandas `category` dtype), passing raw string columns raises a `ValueError`, unlike CatBoost.

### Step 1: Importing (XGBoost)

> 💡 *Beginner Tip:* We import `XGBClassifier` from the `xgboost` library.

In [ ]:
# 1. Importing libraries for XGBoost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)



### Step 2: Loading Dataset

> 💡 *Real-World Context:* Standard Breast Cancer dataset.

In [ ]:
# 2. Loading Dataset
cancer_xgb = load_breast_cancer(as_frame=True)
X_xgb = cancer_xgb.data
y_xgb = cancer_xgb.target
display(X_xgb.head(2))


### Step 3: Split (Stratified Partitioning)

> ⚠️ *Critical Engineering Concept:* Stratified partitioning.

In [ ]:
# 3. Train/Test Split
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb, y_xgb, test_size=0.25, random_state=42, stratify=y_xgb
)
print(f"Training samples: {len(X_train_xgb)} | Testing samples: {len(X_test_xgb)}")


### Step 4: Train (XGBoost 2nd-Order Tree Setup)

> 💡 *Hyperparameter Intuition:* Setting `max_depth=4`, `learning_rate=0.1`, `n_estimators=100`, and `eval_metric='logloss'`.

In [ ]:
# 4. Train: Model Initialization
xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    eval_metric='logloss',
    random_state=42
)
print("XGBoost model initialized:", xgb_clf)


### Step 5: Fit (Exact Greedy Regularized Training)

> 💡 *Mathematical Insight — Why `eval_set=[(X_train, y_train), (X_test, y_test)]` Is Passed:*  
> Passing both datasets inside `eval_set` instructs XGBoost to compute and log binary log-loss after *every single tree* for both training (`validation_0`) and testing (`validation_1`), storing the full learning trajectory inside `xgb_clf.evals_result()` for overfitting diagnostics.

In [ ]:
# 5. Fit: Model Training
xgb_clf.fit(
    X_train_xgb, y_train_xgb,
    eval_set=[(X_train_xgb, y_train_xgb), (X_test_xgb, y_test_xgb)],
    verbose=False
)
print("XGBoost model training complete!")


### Step 6: Predict (Gradient Boosted Inference)

> 💡 *Beginner Tip:* Evaluating test predictions through the regularized gradient boosted forest.

In [ ]:
# 6. Predict: Model Inference
y_pred_xgb = xgb_clf.predict(X_test_xgb)
print("First 10 XGBoost Predictions:", y_pred_xgb[:10])


### Step 7: Quantitative Evaluation (XGBoost Scorecard)

> 💡 *How to Interpret Ensemble Metrics:*  
> Inspect test accuracy alongside clinical Precision, Recall, and F1 score across both Malignant and Benign biopsies.

In [ ]:
# 7. Evaluation
acc_xgb = accuracy_score(y_test_xgb, y_pred_xgb)
cm_xgb = confusion_matrix(y_test_xgb, y_pred_xgb)

print("=" * 60)
print(f"XGBOOST ACCURACY SCORE: {acc_xgb:.4f} ({acc_xgb*100:.2f}%)")
print("=" * 60)
print("CLASSIFICATION REPORT:")
print(classification_report(y_test_xgb, y_pred_xgb, target_names=cancer_xgb.target_names))


### Step 8: Visualization (Log-Loss Learning Curve & Confusion Matrix)

> 💡 *Visual Diagnostics:*  
> 1. **Log-Loss Learning Curve (`evals_result()`):** Compares `'validation_0'` (blue training loss) against `'validation_1'` (red test loss). A healthy model shows both curves descending smoothly together; divergence signals when to enable `early_stopping_rounds`.
> 2. **Confusion Matrix:** Clinical error audit.

In [ ]:
# 8. Visualization: Learning Curve & Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Log-Loss Learning Curve
eval_res = xgb_clf.evals_result()
axes[0].plot(eval_res['validation_0']['logloss'], label='Training Log-Loss', color='#1f77b4', linewidth=2)
axes[0].plot(eval_res['validation_1']['logloss'], label='Testing Log-Loss', color='#e24a33', linewidth=2)
axes[0].set_title("XGBoost Learning Curve (Log-Loss per Iteration)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Boosting Rounds", fontsize=11)
axes[0].set_ylabel("Log-Loss", fontsize=11)
axes[0].legend()

# Plot 2: Confusion Matrix
disp_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=cancer_xgb.target_names)
disp_xgb.plot(cmap="YlGnBu", ax=axes[1], values_format="d", colorbar=False)
axes[1].set_title("XGBoost: Confusion Matrix", fontsize=13, fontweight="bold")
axes[1].grid(False)

plt.tight_layout()
plt.show()


---
# 8. Comprehensive Ensemble Benchmark & Comparison

We now synthesize and compare all seven ensemble paradigms trained and evaluated on the **identical test split** under identical conditions.

### 🔬 Evaluation Dimensions Analyzed:
1. **Accuracy Score:** Overall percentage of correct diagnostic classifications.
2. **Precision:** Out of all cases predicted as malignant, how many were truly malignant? $\frac{TP}{TP + FP}$ *(Critical for minimizing unnecessary patient anxiety and invasive follow-ups).*
3. **Recall (Sensitivity):** Out of all true malignant cancer cases, how many did the ensemble catch? $\frac{TP}{TP + FN}$ *(Critical in medical diagnosis: false negatives are potentially life-threatening!).*
4. **F1-Score:** Harmonic mean of precision and recall: $\frac{2 \times P \times R}{P + R}$.

In [ ]:
# 8.1 Model Performance Leaderboard Synthesis
from sklearn.metrics import precision_score, recall_score, f1_score

models_dict = {
    "Bagging (Decision Trees)": y_pred_bag,
    "Boosting (AdaBoost)": y_pred_ada,
    "Stacking Ensemble": y_pred_stack,
    "Random Forest": y_pred_rf,
    "Gradient Boosting (GBDT)": y_pred_gb,
    "CatBoost": y_pred_cb,
    "XGBoost": y_pred_xgb
}

benchmark_data = []
for name, preds in models_dict.items():
    benchmark_data.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_bag, preds),
        "Precision": precision_score(y_test_bag, preds),
        "Recall": recall_score(y_test_bag, preds),
        "F1-Score": f1_score(y_test_bag, preds)
    })

df_benchmark = pd.DataFrame(benchmark_data).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

print("=" * 80)
print("ENSEMBLE LEARNING BENCHMARK LEADERBOARD (Breast Cancer Wisconsin)")
print("=" * 80)
display(df_benchmark.round(4))


In [ ]:
# 8.2 Visualizations: Ensemble Benchmark Comparison Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df_benchmark))
width = 0.35

rects1 = ax.bar(x - width/2, df_benchmark['Accuracy'], width, label='Accuracy', color='#2b5c8f', edgecolor='black', alpha=0.85)
rects2 = ax.bar(x + width/2, df_benchmark['F1-Score'], width, label='F1-Score', color='#1b9e77', edgecolor='black', alpha=0.85)

ax.set_ylabel('Metric Score', fontsize=11)
ax.set_title('Cross-Ensemble Benchmark: Accuracy & F1-Score Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_benchmark['Model'], rotation=25, ha='right', fontsize=10)
ax.set_ylim(0.90, 1.0)
ax.legend(loc='lower right')

for rect in rects1:
    h = rect.get_height()
    ax.annotate(f'{h:.3f}', xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3),
                textcoords="offset points", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


---
## 9. Comparative Summary: Ensemble Learning Architecture Guide

| Ensemble Method | Paradigm | Base Learners | Variance Reduction? | Bias Reduction? | Primary Hyperparameters | Best Applied When |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Bagging** | Parallel | Homogeneous (unpruned trees) | **High** | Low | `n_estimators`, `max_samples` | High-variance base models needing stabilization; parallel multi-core compute |
| **AdaBoost** | Sequential | Homogeneous (decision stumps) | Low | **High** | `n_estimators`, `learning_rate` | Fast, lightweight bias reduction on clean datasets without extreme label noise |
| **Stacking** | Sequential meta-learning | Heterogeneous (RF, KNN, NB, etc.) | Moderate | **High** | `estimators`, `final_estimator`, `cv` | Blending diverse distinct algorithmic paradigms for maximum predictive competition score |
| **Random Forest** | Parallel | De-correlated trees (random subspace) | **Highest** | Low | `n_estimators`, `max_features='sqrt'` | The universal gold-standard tabular baseline: robust to outliers, zero scaling needed |
| **Gradient Boosting (GBDT)** | Sequential | Shallow regression trees | Moderate | **Highest** | `learning_rate`, `n_estimators`, `subsample` | High-performance tabular data when tuned carefully with cross-validation |
| **CatBoost** | Sequential | Oblivious symmetric trees | Moderate | **Highest** | `iterations`, `depth`, `learning_rate` | Heavy categorical feature columns, fast inference latency requirements |
| **XGBoost** | Sequential | Regularized 2nd-order trees | Moderate | **Highest** | `n_estimators`, `max_depth`, `gamma`, `reg_lambda` | Large-scale production tabular problems, Kaggle competitions, sparse matrix data |

---

### 🏆 Practical Industry Strategy: The 3-Tier Ensemble Ladder
When tackling any new tabular machine learning project in industry:
1. **Tier 1 (The 5-Minute Baseline):** Always start with **Random Forest** (`RandomForestClassifier(n_estimators=100)`). It requires zero feature scaling, handles outliers gracefully, almost never overfits severely, and gives an immediate benchmark.
2. **Tier 2 (The Production Workhorse):** Train **XGBoost** or **CatBoost**. If you have categorical columns (text categories, cities, departments), use CatBoost. If numerical with sparse features, tune XGBoost with early stopping.
3. **Tier 3 (The Competition Finisher):** If you need to squeeze out the final $0.5\%$ accuracy for a Kaggle competition or mission-critical model, take your best Random Forest, XGBoost, CatBoost, and a LightGBM model, and stack them via **StackingClassifier** with Logistic Regression as the meta-learner.

---

### 📝 Top 5 Ensemble Learning Pitfalls & Best Practices:
1. **AdaBoost Noise Sensitivity:** AdaBoost aggressively focuses on misclassified points. If your dataset contains mislabeled noisy data, AdaBoost will waste all its capacity memorizing noise!
2. **Stacking Data Leakage:** Never train a meta-learner on predictions from base models evaluated on their own training data. Always use out-of-fold cross-validation predictions (`StackingClassifier(cv=5)`).
3. **Learning Rate vs. Estimators Tradeoff in Boosting:** Lower learning rate (e.g., $0.01$ vs $0.1$) always requires more estimators ($500$ vs $100$) to reach convergence. Always pair boosting with **early stopping** on a validation set!
4. **Correlated Base Models in Stacking:** Stacking three identical Random Forests yields no improvement. Stacking works only when base models have **uncorrelated error distributions** (e.g., combining a tree with a neural net and a nearest-neighbor model).
5. **Over-interpreting GBDT Feature Importances:** Default Gini importance in tree models is biased toward continuous features with high cardinality. Use **Permutation Importance** or **SHAP values** for clinical/business decisions.